# kernel : ml-dl-nlp
# 파일 read

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
data = pd.read_pickle('data/industry_labeled.pickle')
stock_data = pd.read_csv('data/주가지수크롤링(증감치숫자로).csv')
data.sort_values(by=['산업','날짜'],ascending=[True,True], inplace=True)
#  날짜를 datetime형으로 변환
data["날짜"] = pd.to_datetime(data["날짜"]).dt.date

# 건설, 자동차 헬스케어 분리
build_data = data[data['산업'].values=='건설']
car_data = data[data['산업'].values=='자동차']
health_data = data[data['산업'].values=='헬스케어']

stock_data['날짜']= pd.to_datetime(stock_data['날짜'].astype(str), format="%Y%m%d")
stock_data.info()
# build_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  174 non-null    int64         
 1   날짜          174 non-null    datetime64[ns]
 2   종목명         174 non-null    object        
 3   종가(백만원)     174 non-null    float64       
 4   거래량(천주)     174 non-null    int64         
 5   전일대비        174 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(1)
memory usage: 8.3+ KB


# 감정분석 모델

In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 감정분석 모델 로딩 
# KR-FinBERT : 금융 텍스트에 특화된 한국어 BERT 기반 모델로, 뉴스·리포트·공시 등에서 긍정·부정·중립 감정을 자동으로 분류하는 데 활용
tokenizer = AutoTokenizer.from_pretrained("snunlp/v")
model = AutoModelForSequenceClassification.from_pretrained("snunlp/KR-FinBERT")

def get_sentiment_probs(text):
                                                # 긴 문장은 잘라냄
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).detach().numpy()[0]
    return probs  # [p_neg, p_pos] ex) [0.3, 0.7] 부정 30%, 긍정 70% 

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at snunlp/KR-FinBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# 건설 감정분석(날짜별 통합, 주말은 월요일에 통합)

In [31]:
import pandas as pd
import numpy as np
from datetime import timedelta

def build_sentiment(build_data, stock_data, start_date, end_date):
    # 날짜 범위 계산
    diff_date = (end_date - start_date).days
    check_date = start_date

    # 거래일 리스트 준비
    stock_data['날짜'] = pd.to_datetime(stock_data['날짜']).dt.date
    trading_days = sorted(stock_data['날짜'].unique())

    def map_to_next_trading_day(d):
        # d 이후의 거래일 중 가장 가까운 날 찾기
        future_days = [day for day in trading_days if day > d]
        if len(future_days) == 0:
            return None
        return min(future_days)

    # 최종 결과 DataFrame
    final_df = pd.DataFrame(columns=['거래일', '종목명', 'sentiment', '전일대비'])

    for i in range(diff_date):
        # 건설 뉴스만 추출
        b_news_df = build_data[build_data['날짜'] == check_date]
        if b_news_df.empty:
            check_date += timedelta(days=1)
            continue

        cols = ['날짜', '제목', '본문', '링크', '산업']
        news_df = b_news_df[cols].copy()

        # 뉴스 감정 분석
        news_df['sentiment'] = news_df['제목'].apply(get_sentiment_probs)

        # 하루치 기사별 평균 감정 확률
        daily_sentiment = news_df.groupby('날짜')['sentiment'].apply(
            lambda x: np.mean(np.stack(x), axis=0)
        ).reset_index()

        # 타입 맞추기
        daily_sentiment['날짜'] = pd.to_datetime(daily_sentiment['날짜']).dt.date

        # 주말 뉴스 → 다음 거래일 매핑
        daily_sentiment['거래일'] = daily_sentiment['날짜'].apply(map_to_next_trading_day)

        # 거래일 없는 경우 제외
        daily_sentiment = daily_sentiment.dropna(subset=['거래일'])
        if daily_sentiment.empty:
            check_date += timedelta(days=1)
            continue

        # 건설 종목만 병합
        b_stock_df = stock_data[stock_data['종목명'] == 'KRX 건설']
        daily_sentiment = pd.merge(
            daily_sentiment,
            b_stock_df,
            left_on=daily_sentiment['거래일'].apply(pd.Timestamp), 
            right_on=pd.to_datetime(b_stock_df['날짜']),
            how='left'
        )

        # 원하는 열만 정리
        cols = ['거래일', '종목명', 'sentiment', '종가(백만원)', '전일대비']
        daily_sentiment = daily_sentiment[cols]

        # 결과 누적
        final_df = pd.concat([final_df, daily_sentiment], ignore_index=True)

        check_date += timedelta(days=1)

    # 날짜 기준으로 정렬
    if not final_df.empty:
        final_df = final_df.sort_values(by='거래일').reset_index(drop=True)

        # 이전 행과 전일대비 값 비교 → 중복 제거
        mask = final_df['전일대비'] != final_df['전일대비'].shift(1)
        final_df = final_df[mask]

        return final_df
    else:
        print("결과가 없습니다.")
        return pd.DataFrame()


In [32]:
start_date = pd.to_datetime("2025-12-01")
end_date = pd.to_datetime("2025-12-21")

build_final_df = build_sentiment(build_data, stock_data, start_date, end_date)
print(build_final_df)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


           거래일     종목명                 sentiment   전일대비  종가(백만원)
0   2025-12-02  KRX 건설   [0.5131392, 0.48686075]   9.28   785.34
1   2025-12-03  KRX 건설  [0.51107407, 0.48892602]  24.01   809.35
2   2025-12-04  KRX 건설   [0.5126945, 0.48730552]  -9.54   799.81
3   2025-12-05  KRX 건설    [0.511167, 0.48883304]  36.51   836.32
4   2025-12-08  KRX 건설   [0.5050401, 0.49495983] -13.97   822.35
7   2025-12-09  KRX 건설   [0.51154375, 0.4884562]  -7.25   815.10
8   2025-12-10  KRX 건설    [0.515059, 0.48494095]  -2.82   812.28
9   2025-12-11  KRX 건설   [0.5097892, 0.49021065]  18.91   831.19
10  2025-12-12  KRX 건설    [0.50239193, 0.497608]  38.62   869.81
11  2025-12-15  KRX 건설   [0.5017834, 0.49821663] -30.13   839.68
14  2025-12-16  KRX 건설   [0.5105873, 0.48941278] -22.74   816.94
15  2025-12-17  KRX 건설  [0.50931656, 0.49068338]  13.58   830.52
16  2025-12-18  KRX 건설  [0.50285375, 0.49714628] -11.06   819.46
17  2025-12-19  KRX 건설   [0.5091524, 0.49084765]   1.14   820.60
18  2025-12-22  KRX 건설   

# 자동차 감정분석

In [14]:
def car_sentiment(car_data, stock_data, start_date, end_date):
    # 날짜 범위 계산
    diff_date = (end_date - start_date).days
    check_date = start_date

    # 거래일 리스트 준비
    stock_data['날짜'] = pd.to_datetime(stock_data['날짜']).dt.date
    trading_days = sorted(stock_data['날짜'].unique())

    def map_to_next_trading_day(d):
        # d 이후의 거래일 중 가장 가까운 날 찾기
        future_days = [day for day in trading_days if day > d]
        if len(future_days) == 0:
            return None
        return min(future_days)

    # 최종 결과 DataFrame
    final_df = pd.DataFrame(columns=['거래일', '종목명', 'sentiment', '종가(백만원)', '전일대비'])

    for i in range(diff_date):
        # 자동차 뉴스만 추출
        c_news_df = car_data[car_data['날짜'] == check_date]
        if c_news_df.empty:
            check_date += timedelta(days=1)
            continue

        cols = ['날짜', '제목', '본문', '링크', '산업']
        news_df = c_news_df[cols].copy()

        # 뉴스 감정 분석
        news_df['sentiment'] = news_df['제목'].apply(get_sentiment_probs)

        # 하루치 기사별 평균 감정 확률
        daily_sentiment = news_df.groupby('날짜')['sentiment'].apply(
            lambda x: np.mean(np.stack(x), axis=0)
        ).reset_index()

        # 타입 맞추기
        daily_sentiment['날짜'] = pd.to_datetime(daily_sentiment['날짜']).dt.date

        # 주말 뉴스 → 다음 거래일 매핑
        daily_sentiment['거래일'] = daily_sentiment['날짜'].apply(map_to_next_trading_day)

        # 거래일 없는 경우 제외
        daily_sentiment = daily_sentiment.dropna(subset=['거래일'])
        if daily_sentiment.empty:
            check_date += timedelta(days=1)
            continue

        # 자동차 종목만 병합
        c_stock_df = stock_data[stock_data['종목명'] == 'KRX 자동차']
        daily_sentiment = pd.merge(
            daily_sentiment,
            c_stock_df,
            left_on=daily_sentiment['거래일'].apply(pd.Timestamp), 
            right_on=pd.to_datetime(c_stock_df['날짜']),
            how='left'
        )

        # 원하는 열만 정리
        cols = ['거래일', '종목명', 'sentiment', '전일대비']
        daily_sentiment = daily_sentiment[cols]

        # 결과 누적
        final_df = pd.concat([final_df, daily_sentiment], ignore_index=True)

        check_date += timedelta(days=1)

    # 날짜 기준으로 정렬
    if not final_df.empty:
        final_df = final_df.sort_values(by='거래일').reset_index(drop=True)

        # 이전 행과 전일대비 값 비교 → 중복 제거
        mask = final_df['전일대비'] != final_df['전일대비'].shift(1)
        final_df = final_df[mask]

        return final_df
    else:
        print("결과가 없습니다.")
        return pd.DataFrame()


In [15]:
start_date = pd.to_datetime("2025-12-01")
end_date = pd.to_datetime("2025-12-21")

car_final_df = car_sentiment(build_data, stock_data, start_date, end_date)
print(car_final_df)

           거래일      종목명                 sentiment    전일대비
0   2025-12-02  KRX 자동차   [0.5240867, 0.47591317]   81.63
1   2025-12-03  KRX 자동차   [0.5234777, 0.47652236]   39.31
2   2025-12-04  KRX 자동차    [0.5276376, 0.4723625]   71.24
3   2025-12-05  KRX 자동차   [0.5211304, 0.47886968]  120.77
4   2025-12-08  KRX 자동차   [0.5249894, 0.47501054]   -2.78
7   2025-12-09  KRX 자동차    [0.5207479, 0.4792522]  -45.02
8   2025-12-10  KRX 자동차  [0.52587986, 0.47412014]  -12.24
9   2025-12-11  KRX 자동차   [0.5223742, 0.47762573]  -30.82
10  2025-12-12  KRX 자동차   [0.49801356, 0.5019864]   62.99
11  2025-12-15  KRX 자동차  [0.51204675, 0.48795336]  -46.82
14  2025-12-16  KRX 자동차   [0.51963323, 0.4803668]  -47.17
15  2025-12-17  KRX 자동차   [0.51139224, 0.4886077]   23.76
16  2025-12-18  KRX 자동차    [0.51887697, 0.481123]  -50.34
17  2025-12-19  KRX 자동차     [0.5244281, 0.475572]   56.76
18  2025-12-22  KRX 자동차   [0.5163478, 0.48365214]    7.47


# 헬스케어 감정분석

In [16]:
def health_sentiment(car_data, stock_data, start_date, end_date):
    # 날짜 범위 계산
    diff_date = (end_date - start_date).days
    check_date = start_date

    # 거래일 리스트 준비
    stock_data['날짜'] = pd.to_datetime(stock_data['날짜']).dt.date
    trading_days = sorted(stock_data['날짜'].unique())

    def map_to_next_trading_day(d):
        # d 이후의 거래일 중 가장 가까운 날 찾기
        future_days = [day for day in trading_days if day > d]
        if len(future_days) == 0:
            return None
        return min(future_days)

    # 최종 결과 DataFrame
    final_df = pd.DataFrame(columns=['거래일', '종목명', 'sentiment', '종가(백만원)', '전일대비'])

    for i in range(diff_date):
        # 건설 뉴스만 추출
        h_news_df = health_data[health_data['날짜'] == check_date]
        if h_news_df.empty:
            check_date += timedelta(days=1)
            continue

        cols = ['날짜', '제목', '본문', '링크', '산업']
        news_df = h_news_df[cols].copy()

        # 뉴스 감정 분석
        news_df['sentiment'] = news_df['제목'].apply(get_sentiment_probs)

        # 하루치 기사별 평균 감정 확률
        daily_sentiment = news_df.groupby('날짜')['sentiment'].apply(
            lambda x: np.mean(np.stack(x), axis=0)
        ).reset_index()

        # 타입 맞추기
        daily_sentiment['날짜'] = pd.to_datetime(daily_sentiment['날짜']).dt.date

        # 주말 뉴스 → 다음 거래일 매핑
        daily_sentiment['거래일'] = daily_sentiment['날짜'].apply(map_to_next_trading_day)

        # 거래일 없는 경우 제외
        daily_sentiment = daily_sentiment.dropna(subset=['거래일'])
        if daily_sentiment.empty:
            check_date += timedelta(days=1)
            continue

        # 헬스케어 종목만 병합
        h_stock_df = stock_data[stock_data['종목명'] == 'KRX 헬스케어']
        daily_sentiment = pd.merge(
            daily_sentiment,
            h_stock_df,
            left_on=daily_sentiment['거래일'].apply(pd.Timestamp), 
            right_on=pd.to_datetime(h_stock_df['날짜']),
            how='left'
        )

        # 원하는 열만 정리
        cols = ['거래일', '종목명', 'sentiment', '전일대비']
        daily_sentiment = daily_sentiment[cols]

        # 결과 누적
        final_df = pd.concat([final_df, daily_sentiment], ignore_index=True)

        check_date += timedelta(days=1)

    # 날짜 기준으로 정렬
    if not final_df.empty:
        final_df = final_df.sort_values(by='거래일').reset_index(drop=True)

        # 이전 행과 전일대비 값 비교 → 중복 제거
        mask = final_df['전일대비'] != final_df['전일대비'].shift(1)
        final_df = final_df[mask]

        return final_df
    else:
        print("결과가 없습니다.")
        return pd.DataFrame()


In [17]:
start_date = pd.to_datetime("2025-12-01")
end_date = pd.to_datetime("2025-12-21")

health_final_df = health_sentiment(health_data, stock_data, start_date, end_date)
print(health_final_df)

           거래일       종목명                 sentiment    전일대비
0   2025-12-02  KRX 헬스케어  [0.48629072, 0.51370925]  -37.20
1   2025-12-03  KRX 헬스케어   [0.47921807, 0.5207819]   -1.02
2   2025-12-04  KRX 헬스케어    [0.4850371, 0.5149629]   12.03
3   2025-12-05  KRX 헬스케어   [0.47937867, 0.5206213] -130.37
4   2025-12-08  KRX 헬스케어   [0.4844076, 0.51559246]  -54.21
7   2025-12-09  KRX 헬스케어    [0.4873974, 0.5126027]   51.86
8   2025-12-10  KRX 헬스케어  [0.48321566, 0.51678437]   41.44
9   2025-12-11  KRX 헬스케어  [0.48744372, 0.51255625]   13.63
10  2025-12-12  KRX 헬스케어   [0.4857628, 0.51423717]  -48.01
11  2025-12-15  KRX 헬스케어  [0.48630974, 0.51369023]   62.48
14  2025-12-16  KRX 헬스케어   [0.48365656, 0.5163435]  -46.23
15  2025-12-17  KRX 헬스케어  [0.48457682, 0.51542324]  -85.00
16  2025-12-18  KRX 헬스케어   [0.48461714, 0.5153829]  -20.52
17  2025-12-19  KRX 헬스케어  [0.48710293, 0.51289713]   85.57
18  2025-12-22  KRX 헬스케어    [0.48938605, 0.510614]  -44.41


# 리스크 지표(사용안함)

In [48]:
import pandas as pd
import numpy as np

# 예시: 일별 수익률 데이터
df['return'] = df['전일대비']/(df['종가(백만원)']-df['전일대비'])

# 1. 변동성 (20일 롤링) ex) 0.015 : 하루 수익률이 ±1.5% 정도 등락할 수 있음
df['volatility'] = df['return'].rolling(3).std()

# 2. VaR (95% 신뢰수준) ex) -0.015 : 95%의 확률로 하루 손실은 -1.5% 이내일 수 있음
confidence_level = 0.95
df['VaR_95'] = df['return'].rolling(3).apply(
    lambda x: np.percentile(x, (1-confidence_level)*100)
)

# 3. CVaR (Expected Shortfall) ex) -0.03 : VaR을 넘어서는 극단상황에서는 평균적으로 -3% 정도 손해가 발생함
df['CVaR_95'] = df['return'].rolling(3).apply(
    lambda x: x[x <= np.percentile(x, (1-confidence_level)*100)].mean()
)

# 4. Drawdown # 최대낙폭
df['cum_max'] = df['종가(백만원)'].cummax() # 종가의 누적 최고점
df['drawdown'] = (df['종가(백만원)'] - df['cum_max']) / df['cum_max'] # 고점 대비 하락폭 ex) -0.15 : 초고점 대비 15% 하락



,거래일,종목명,sentiment,전일대비,종가(백만원),sent_neg,sent_pos,return,volatility,VaR_95,CVaR_95,cum_max,drawdown
0,2025-12-02,KRX 건설,"[0.5131392, 0.48686075]",9.28,785.34,0.513139,0.486861,0.011958,NaN,NaN,NaN,785.34,0.000000
1,2025-12-03,KRX 건설,"[0.51107407, 0.48892602]",24.01,809.35,0.511074,0.488926,0.030573,NaN,NaN,NaN,809.35,0.000000
2,2025-12-04,KRX 건설,"[0.5126945, 0.48730552]",-9.54,799.81,0.512694,0.487306,-0.011787,0.021232,-0.009413,-0.011787,809.35,-0.011787
3,2025-12-05,KRX 건설,"[0.511167, 0.48883304]",36.51,836.32,0.511167,0.488833,0.045648,0.029778,-0.007551,-0.011787,836.32,0.000000
4,2025-12-08,KRX 건설,"[0.5050401, 0.49495983]",-13.97,822.35,0.505040,0.494960,-0.016704,0.034667,-0.016212,-0.016704,836.32,-0.016704
7,2025-12-09,KRX 건설,"[0.51154375, 0.4884562]",-7.25,815.10,0.511544,0.488456,-0.008816,0.033952,-0.015915,-0.016704,836.32,-0.025373
8,2025-12-10,KRX 건설,"[0.515059, 0.48494095]",-2.82,812.28,0.515059,0.484941,-0.003460,0.006662,-0.015915,-0.016704,836.32,-0.028745
9,2025-12-11,KRX 건설,"[0.5097892, 0.49021065]",18.91,831.19,0.509789,0.490211,0.023280,0.017194,-0.008281,-0.008816,836.32,-0.006134
10,2025-12-12,KRX 건설,"[0.50239193, 0.497608]",38.62,869.81,0.502392,0.497608,0.046464,0.024983,-0.000786,-0.003460,869.81,0.000000
11,2025-12-15,KRX 건설,"[0.5017834, 0.49821663]",-30.13,839.68,0.501783,0.498217,-0.034640,0.041773,-0.028848,-0.034640,869.81,-0.034640


In [55]:
df.drop(['sentiment'], axis =1, inplace=True)
df

,거래일,종목명,전일대비,종가(백만원),sent_neg,sent_pos,return,volatility,VaR_95,CVaR_95,cum_max,drawdown
0,2025-12-02,KRX 건설,9.28,785.34,0.513139,0.486861,0.011958,NaN,NaN,NaN,785.34,0.000000
1,2025-12-03,KRX 건설,24.01,809.35,0.511074,0.488926,0.030573,NaN,NaN,NaN,809.35,0.000000
2,2025-12-04,KRX 건설,-9.54,799.81,0.512694,0.487306,-0.011787,0.021232,-0.009413,-0.011787,809.35,-0.011787
3,2025-12-05,KRX 건설,36.51,836.32,0.511167,0.488833,0.045648,0.029778,-0.007551,-0.011787,836.32,0.000000
4,2025-12-08,KRX 건설,-13.97,822.35,0.505040,0.494960,-0.016704,0.034667,-0.016212,-0.016704,836.32,-0.016704
7,2025-12-09,KRX 건설,-7.25,815.10,0.511544,0.488456,-0.008816,0.033952,-0.015915,-0.016704,836.32,-0.025373
8,2025-12-10,KRX 건설,-2.82,812.28,0.515059,0.484941,-0.003460,0.006662,-0.015915,-0.016704,836.32,-0.028745
9,2025-12-11,KRX 건설,18.91,831.19,0.509789,0.490211,0.023280,0.017194,-0.008281,-0.008816,836.32,-0.006134
10,2025-12-12,KRX 건설,38.62,869.81,0.502392,0.497608,0.046464,0.024983,-0.000786,-0.003460,869.81,0.000000
11,2025-12-15,KRX 건설,-30.13,839.68,0.501783,0.498217,-0.034640,0.041773,-0.028848,-0.034640,869.81,-0.034640


In [60]:
df=df.drop(['sent_neg', 'sent_pos'],axis=1)
df

,거래일,종목명,전일대비,종가(백만원),return,volatility,VaR_95,CVaR_95,cum_max,drawdown
0,2025-12-02,KRX 건설,9.28,785.34,0.011958,NaN,NaN,NaN,785.34,0.000000
1,2025-12-03,KRX 건설,24.01,809.35,0.030573,NaN,NaN,NaN,809.35,0.000000
2,2025-12-04,KRX 건설,-9.54,799.81,-0.011787,0.021232,-0.009413,-0.011787,809.35,-0.011787
3,2025-12-05,KRX 건설,36.51,836.32,0.045648,0.029778,-0.007551,-0.011787,836.32,0.000000
4,2025-12-08,KRX 건설,-13.97,822.35,-0.016704,0.034667,-0.016212,-0.016704,836.32,-0.016704
7,2025-12-09,KRX 건설,-7.25,815.10,-0.008816,0.033952,-0.015915,-0.016704,836.32,-0.025373
8,2025-12-10,KRX 건설,-2.82,812.28,-0.003460,0.006662,-0.015915,-0.016704,836.32,-0.028745
9,2025-12-11,KRX 건설,18.91,831.19,0.023280,0.017194,-0.008281,-0.008816,836.32,-0.006134
10,2025-12-12,KRX 건설,38.62,869.81,0.046464,0.024983,-0.000786,-0.003460,869.81,0.000000
11,2025-12-15,KRX 건설,-30.13,839.68,-0.034640,0.041773,-0.028848,-0.034640,869.81,-0.034640


# 여기서 부터는 작업중_BiLSTM딥러닝(날릴수도?)

In [24]:
import torch.nn as nn
import numpy as np

# sentiment를 2개 컬럼으로 분리
build_final_df[['sent_neg','sent_pos']] = pd.DataFrame(
                                    build_final_df['sentiment'].tolist(), index=build_final_df.index)

# 입력 특징 X (sentiment만 사용하거나 sentiment+전일대비 같이 사용 가능)
X = build_final_df[['sent_neg','sent_pos']].values
# 타겟 y (전일대비 → 상승/하락 분류라면 0/1로 변환)
y = (build_final_df['전일대비'] > 0).astype(int).values   # 상승=1, 하락=0


class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim*2, output_dim)  # 양방향이므로 hidden_dim*2

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # 마지막 타임스텝 출력
        return out


In [25]:
import torch

# 모델 정의
model = BiLSTM(input_dim=X.shape[1], hidden_dim=32, output_dim=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

losses = []

# 학습 루프
epochs = 100
for epoch in range(epochs):
    outputs = model(X_tensor)          # forward
    loss = criterion(outputs, y_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item()) 
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

NameError: name 'X_tensor' is not defined

# lightgbm 모델

In [34]:
# 1. 라이브러리 불러오기
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb

# 2. 데이터 준비 (예시: 감정 점수 + 주가 데이터)
# sentiment_neg, sentiment_pos, 전일대비 등 특징(feature) 포함
# target: 상승(1), 하락(0)

# 건설
df = build_final_df
df[['sent_neg','sent_pos']] = pd.DataFrame(
                                    build_final_df['sentiment'].tolist(), index=build_final_df.index)
# # 자동차
# df = car_final_df
# df[['sent_neg','sent_pos']] = pd.DataFrame(
#                                     car_final_df['sentiment'].tolist(), index=build_final_df.index)

# # 헬스케어
# df = health_final_df
# df[['sent_neg','sent_pos']] = pd.DataFrame(
#                                     health_final_df['sentiment'].tolist(), index=build_final_df.index)



X = df[['sent_neg', 'sent_pos', '전일대비']]   # 입력 특징
y = (df['전일대비'] > 0).astype(int)                     # 타깃: 상승=1, 하락=0

# 3. 학습/검증 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # 시계열이면 shuffle=False 권장
)

# 4. LightGBM 데이터셋 생성
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# 5. 하이퍼파라미터 설정
params = {
    'objective': 'binary',        # 이진 분류
    'metric': 'binary_error',     # 방향성 정확도
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

# 6. 모델 학습
model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],
    num_boost_round=200,
    #early_stopping_rounds=20
)

# 7. 예측
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

# 8. 평가
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.6666666666666666
Confusion Matrix:
 [[0 1]
 [0 2]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.67      1.00      0.80         2

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3

